# LLM Inference Optimization Benchmark Suite
### Project 3 of 4 — GPU/ML Systems Engineer Portfolio

This notebook systematically benchmarks **Llama-3.2-3B-Instruct** across every major inference optimization axis.

| Phase | What it measures |
|---|---|
| 1 | Environment + baseline setup |
| 2 | FP16 vs INT8 vs GPTQ vs AWQ (raw HF `generate()`) |
| 3 | Accuracy tradeoff (perplexity + MMLU) per precision |
| 4 | vLLM engine benchmark (PagedAttention, chunked prefill) |
| 5 | vLLM + quantization combined |
| 6 | Locust concurrency sweep (1→32 users) |
| 7 | GPU telemetry (DCGM-equivalent via pynvml) |
| 8 | Cost-per-token report + instance recommendation |
| 9 (optional) | TensorRT-LLM compilation (requires A100) |

**Runtime requirement:** Colab T4 (free tier) for Phases 1-8. Phase 9 requires Colab Pro with an A100 runtime — gated behind a GPU architecture check so the notebook degrades gracefully on T4.

**Before running:** Set `HF_TOKEN` in Colab's Secrets (key icon, left sidebar) since Llama-3.2 is a gated model requiring HuggingFace license acceptance.

## Phase 1 — Environment Setup

In [ ]:
# Cell 1.1 — Verify GPU and check architecture
# WHY this check runs first, before any pip install: T4 (compute capability
# 7.5, Turing) and A100 (compute capability 8.0, Ampere) support different
# quantization kernels and dtypes (e.g. bfloat16, fp8 KV cache). Every later
# cell that branches behavior by GPU type reads GPU_ARCH set here.
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "No GPU detected -- set Runtime > Change runtime type > GPU"
GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)
GPU_ARCH = "ampere_or_newer" if GPU_CAPABILITY[0] >= 8 else "turing_or_older"
print(f"GPU: {GPU_NAME}, Compute Capability: {GPU_CAPABILITY}, Arch class: {GPU_ARCH}")

In [2]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url(
    "https://repo.anaconda.com/miniconda/Miniconda3-py311_24.11.1-0-Linux-x86_64.sh",
    sha256=None
)

⏬ Downloading https://repo.anaconda.com/miniconda/Miniconda3-py311_24.11.1-0-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:18
🔁 Restarting kernel...


In [2]:
# Cell 1.2.2 — Verify conda took over the kernel, then install everything into base
import condacolab
condacolab.check()

import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["BUILD_CUDA_EXT"] = "0"

!pip install -q torch==2.3.1 numpy==1.26.4 gekko pandas==2.2.2
!pip install -q auto-gptq==0.7.1 --no-build-isolation --extra-index-url https://huggingface.github.io/autogptq-index/whl/cu121/
!pip install -q --no-build-isolation \
    transformers==4.42.3 \
    accelerate==0.31.0 \
    sentencepiece==0.2.0 \
    safetensors==0.4.3 \
    bitsandbytes==0.43.1 \
    vllm==0.5.0 \
    ray==2.31.0 \
    locust==2.29.1 \
    matplotlib==3.9.0 \
    plotly==5.22.0 \
    lm-eval==0.4.3 \
    evaluate==0.4.2 \
    datasets==2.20.0 \
    scikit-learn==1.5.0 \
    nvidia-ml-py==12.560.30 \
    pynvml==11.5.0 \
    fastapi==0.111.0 \
    uvicorn==0.30.1 \
    httpx==0.27.0 \
    aiohttp==3.9.5 \
    pydantic==2.7.4 \
    tqdm==4.66.4 \
    python-dotenv==1.0.1 \
    rich==13.7.1
!pip install -q --no-deps optimum==1.20.0
!pip install -q --no-deps autoawq==0.2.7.post3
!pip install -q --no-deps autoawq-kernels

✨🍰✨ Everything looks OK!
/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.2/779.2 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 59.3 MB/s eta 0:00

In [3]:
%%writefile check_env.py
import transformers, awq, vllm
from importlib.metadata import version as pkg_version

print('transformers', transformers.__version__)
print('optimum', pkg_version('optimum'))
print('awq', awq.__version__)
print('vllm', vllm.__version__)

import optimum
from optimum.gptq import GPTQQuantizer
print('optimum.gptq import OK')

Writing check_env.py


In [5]:
!pip install -q "accelerate>=1.2.0"

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autoawq 0.2.7.post3 requires transformers>=4.45.0, but you have transformers 4.42.3 which is incompatible.


In [7]:
!pip install -q "peft==0.11.1" --no-deps

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [8]:
# Cell 1.2.4 — Run it
!python check_env.py

/bin/bash: /usr/local/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/usr/local/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (6.0.0.post1)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(
transformers 4.42.3
optimum 1.20.0
awq 0.2.7.post3
vllm 0.5.0
CUDA extension not installed.
CUDA extension not installed.
optimum.gptq import OK


In [ ]:
# !git clone https://github.com/arkanathroy/llm-inference-benchmark-suite.git
# %cd llm-inference-benchmark-suite

In [ ]:
# Cell 1.3 — Authenticate with HuggingFace (Llama-3.2 is a gated model)
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print("Authenticated.")

In [ ]:
# Cell 1.4 — Mount Drive for persistent benchmark artifact storage
# WHY Drive and not the local Colab runtime filesystem: /content is wiped on
# every session disconnect. Quantized model checkpoints (GPTQ/AWQ) take
# 5-15 minutes to calibrate -- losing them on a disconnect is expensive.
# Saving to Drive means Phase 2's quantized checkpoints persist across
# sessions and don't need to be regenerated if the runtime disconnects
# partway through the notebook.
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = "/content/drive/MyDrive/llm-inference-benchmark-suite"
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(f"{BASE_DIR}/benchmarks", exist_ok=True)
os.makedirs(f"{BASE_DIR}/gptq_model", exist_ok=True)
os.makedirs(f"{BASE_DIR}/awq_model", exist_ok=True)

import sys
sys.path.insert(0, "src")

## Phase 2 — Precision Variant Benchmarking (Raw HuggingFace `generate()`)

This phase measures each of the four precision variants using the plain `transformers` `.generate()` call — deliberately WITHOUT vLLM — to isolate the pure quantization effect (memory + speed + accuracy) from serving-engine effects (batching, PagedAttention), which are layered on separately in Phase 4-5. Conflating the two would make it impossible to tell whether a throughput gain came from "quantization made compute cheaper" or "vLLM batched requests better."

In [ ]:
# Cell 2.1 — Load FP16 baseline
import sys; sys.path.insert(0, "src")
from config import CONFIG
from quantize import load_fp16, model_memory_footprint_mb
import torch, time, gc

model_fp16, tokenizer = load_fp16()
mem_fp16 = model_memory_footprint_mb(model_fp16)
print(f"FP16 model memory footprint: {mem_fp16:.1f} MB")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")

In [ ]:
# Cell 2.2 — Single-request latency benchmark helper
# WHY warmup runs before timed runs: the FIRST forward pass on any model
# pays a one-time cost for CUDA kernel JIT compilation/caching and memory
# allocator warmup -- including it in timed averages systematically
# overstates steady-state latency by as much as 2-3x on the very first call.
@torch.no_grad()
def benchmark_single_request(model, tokenizer, prompt, max_new_tokens=256, n_warmup=2, n_trials=5):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    for _ in range(n_warmup):
        _ = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                            pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    latencies = []
    for _ in range(n_trials):
        torch.cuda.synchronize()
        t0 = time.time()
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
        torch.cuda.synchronize()
        latencies.append(time.time() - t0)
    n_generated = out.shape[1] - inputs['input_ids'].shape[1]
    return {
        "mean_latency_s": sum(latencies) / len(latencies),
        "tokens_per_sec": n_generated / (sum(latencies) / len(latencies)),
        "n_generated_tokens": n_generated,
    }

test_prompt = "Explain the difference between TCP and UDP in three sentences."
fp16_bench = benchmark_single_request(model_fp16, tokenizer, test_prompt)
print(fp16_bench)

In [ ]:
# Cell 2.3 — Free FP16 model before loading INT8 (T4 has only 16GB VRAM)
# WHY explicit del + gc.collect + empty_cache, not just reassigning the
# variable: Python's reference counting alone does not immediately free
# CUDA memory held by torch tensors if any stray reference exists (e.g. in
# the 'out' variable from Cell 2.2) -- torch.cuda.empty_cache() only returns
# memory to the OS driver, it does not free tensors still referenced by
# Python; gc.collect() must run first to actually drop those references.
del model_fp16, out
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1024**2:.1f} MB")

In [ ]:
# Cell 2.4 — Load and benchmark INT8 (bitsandbytes)
from quantize import load_int8
model_int8, _ = load_int8()
mem_int8 = model_memory_footprint_mb(model_int8)
int8_bench = benchmark_single_request(model_int8, tokenizer, test_prompt)
print(f"INT8 memory footprint: {mem_int8:.1f} MB  ({mem_int8/mem_fp16*100:.1f}% of FP16)")
print(int8_bench)
del model_int8
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 2.5 — Calibrate and quantize GPTQ (one-time, ~5-10 min on T4)
# NOTE: This cell only needs to run ONCE per Colab session series -- the
# quantized checkpoint is saved to Drive (BASE_DIR), so on a reconnect you
# can skip this cell and go straight to Cell 2.6's load_gptq() call.
import os
from quantize import quantize_gptq
gptq_save_dir = f"{BASE_DIR}/gptq_model"
if not os.path.exists(f"{gptq_save_dir}/quantize_config.json"):
    gptq_meta = quantize_gptq(save_dir=gptq_save_dir)
    print(gptq_meta)
else:
    print("GPTQ checkpoint already exists on Drive -- skipping calibration.")

In [ ]:
# Cell 2.6 — Load and benchmark GPTQ
from quantize import load_gptq
model_gptq, _ = load_gptq(save_dir=gptq_save_dir)
mem_gptq = model_memory_footprint_mb(model_gptq)
gptq_bench = benchmark_single_request(model_gptq, tokenizer, test_prompt)
print(f"GPTQ memory footprint: {mem_gptq:.1f} MB  ({mem_gptq/mem_fp16*100:.1f}% of FP16)")
print(gptq_bench)
del model_gptq
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 2.7 — Calibrate and quantize AWQ (one-time, ~5-10 min on T4)
from quantize import quantize_awq
awq_save_dir = f"{BASE_DIR}/awq_model"
if not os.path.exists(f"{awq_save_dir}/config.json"):
    awq_meta = quantize_awq(save_dir=awq_save_dir)
    print(awq_meta)
else:
    print("AWQ checkpoint already exists on Drive -- skipping calibration.")

In [ ]:
# Cell 2.8 — Load and benchmark AWQ
from quantize import load_awq
model_awq, _ = load_awq(save_dir=awq_save_dir)
mem_awq = model_memory_footprint_mb(model_awq)
awq_bench = benchmark_single_request(model_awq, tokenizer, test_prompt)
print(f"AWQ memory footprint: {mem_awq:.1f} MB  ({mem_awq/mem_fp16*100:.1f}% of FP16)")
print(awq_bench)
del model_awq
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 2.9 — Consolidate Phase 2 results
import pandas as pd
phase2_results = pd.DataFrame([
    {"precision": "fp16", "memory_mb": mem_fp16, **fp16_bench},
    {"precision": "int8", "memory_mb": mem_int8, **int8_bench},
    {"precision": "gptq", "memory_mb": mem_gptq, **gptq_bench},
    {"precision": "awq",  "memory_mb": mem_awq,  **awq_bench},
])
phase2_results["memory_reduction_pct"] = (1 - phase2_results["memory_mb"] / mem_fp16) * 100
phase2_results["speedup_vs_fp16"] = phase2_results["tokens_per_sec"] / fp16_bench["tokens_per_sec"]
phase2_results.to_csv(f"{BASE_DIR}/benchmarks/phase2_precision_comparison.csv", index=False)
phase2_results

## Phase 3 — Accuracy Tradeoff Analysis (Perplexity + MMLU)

Each precision variant is re-loaded (one at a time, to respect T4's 16GB VRAM ceiling) and scored on two complementary metrics — see `src/evaluate.py` docstring for why both are needed.

In [ ]:
# Cell 3.1 — Evaluate FP16 baseline (reference point for all deltas)
from evaluate import compute_perplexity, evaluate_mmlu_subset
from quantize import load_fp16
model_fp16, tokenizer = load_fp16()
ppl_fp16 = compute_perplexity(model_fp16, tokenizer)
mmlu_fp16 = evaluate_mmlu_subset(model_fp16, tokenizer)
print(f"FP16 -- Perplexity: {ppl_fp16:.3f}, MMLU accuracy: {mmlu_fp16['accuracy']:.3f}")
del model_fp16
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 3.2 — Evaluate INT8
from quantize import load_int8
model_int8, _ = load_int8()
ppl_int8 = compute_perplexity(model_int8, tokenizer)
mmlu_int8 = evaluate_mmlu_subset(model_int8, tokenizer)
print(f"INT8 -- Perplexity: {ppl_int8:.3f}, MMLU accuracy: {mmlu_int8['accuracy']:.3f}")
del model_int8
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 3.3 — Evaluate GPTQ
from quantize import load_gptq
model_gptq, _ = load_gptq(save_dir=gptq_save_dir)
ppl_gptq = compute_perplexity(model_gptq, tokenizer)
mmlu_gptq = evaluate_mmlu_subset(model_gptq, tokenizer)
print(f"GPTQ -- Perplexity: {ppl_gptq:.3f}, MMLU accuracy: {mmlu_gptq['accuracy']:.3f}")
del model_gptq
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 3.4 — Evaluate AWQ
from quantize import load_awq
model_awq, _ = load_awq(save_dir=awq_save_dir)
ppl_awq = compute_perplexity(model_awq, tokenizer)
mmlu_awq = evaluate_mmlu_subset(model_awq, tokenizer)
print(f"AWQ -- Perplexity: {ppl_awq:.3f}, MMLU accuracy: {mmlu_awq['accuracy']:.3f}")
del model_awq
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 3.5 — Consolidate accuracy results and merge with Phase 2 speed/memory
from evaluate import accuracy_delta_report
accuracy_rows = [
    {"precision": "fp16", "perplexity": ppl_fp16, "mmlu_accuracy": mmlu_fp16["accuracy"]},
    {"precision": "int8", "perplexity": ppl_int8, "mmlu_accuracy": mmlu_int8["accuracy"]},
    {"precision": "gptq", "perplexity": ppl_gptq, "mmlu_accuracy": mmlu_gptq["accuracy"]},
    {"precision": "awq",  "perplexity": ppl_awq,  "mmlu_accuracy": mmlu_awq["accuracy"]},
]
phase3_results = pd.DataFrame(accuracy_rows)

baseline = accuracy_rows[0]
for row in accuracy_rows[1:]:
    delta = accuracy_delta_report(baseline, row)
    print(row["precision"], delta)

# Merge Phase 2 (speed/memory) with Phase 3 (accuracy) into the master
# tradeoff table used for the Memory vs Accuracy vs Throughput scatter plot.
tradeoff_table = phase2_results.merge(phase3_results, on="precision")
tradeoff_table.to_csv(f"{BASE_DIR}/benchmarks/quantization_tradeoff_table.csv", index=False)
tradeoff_table

In [ ]:
# Cell 3.6 — Plot the 3-way tradeoff
import sys; sys.path.insert(0, "src")
from plotting import plot_accuracy_vs_memory
plot_df = tradeoff_table.rename(columns={"tokens_per_sec": "throughput_tok_s"})
fig = plot_accuracy_vs_memory(plot_df, f"{BASE_DIR}/benchmarks/accuracy_vs_memory.png")
fig.show()

## Phase 4 — vLLM Engine Benchmark (PagedAttention + Continuous Batching)

Moves from raw HuggingFace `generate()` to vLLM's `AsyncLLMEngine`, isolating the serving-engine contribution (PagedAttention KV cache management, continuous batching, chunked prefill) on top of the FP16 baseline before combining it with quantization in Phase 5.

See `src/config.py::VLLMConfig` for full parameter rationale — every value used below (`gpu_memory_utilization=0.85`, `block_size=16`, `max_num_seqs=32`, `max_num_batched_tokens=8192`, `enable_chunked_prefill=True`) is explained there.

In [ ]:
# Cell 4.1 — Initialize vLLM AsyncLLMEngine with FP16
# WHY AsyncLLMEngine (in-process) here rather than the HTTP server used in
# Phase 6: this phase needs precise TTFT measurement without HTTP/JSON
# serialization overhead confounding the numbers -- direct engine access
# gives token-by-token callback timestamps.
from vllm import AsyncLLMEngine, SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm_server import build_engine_args
import asyncio

engine_args_dict = build_engine_args(quantization=None)
engine_args = AsyncEngineArgs(**engine_args_dict)
vllm_engine_fp16 = AsyncLLMEngine.from_engine_args(engine_args)
print("vLLM FP16 engine initialized.")

In [ ]:
# Cell 4.2 — Generate benchmark prompt set (shared across ALL Phase 4-6 tests)
from prompt_generator import generate_prompt_batch, prompt_length_distribution_report
from transformers import AutoTokenizer
hf_tokenizer = AutoTokenizer.from_pretrained(CONFIG["model"].model_id)
benchmark_prompts = generate_prompt_batch(n_prompts=64, tokenizer=hf_tokenizer, seed=42)
print(prompt_length_distribution_report(benchmark_prompts, hf_tokenizer))

In [ ]:
# Cell 4.3 — Async inference wrapper capturing TTFT via token-stream callback
from benchmark_runner import RequestResult
import time

async def vllm_infer(engine, prompt, request_id, max_tokens=256):
    sampling_params = SamplingParams(temperature=0.0, max_tokens=max_tokens)
    t0 = time.time()
    ttft = None
    n_tokens = 0
    final_output = None
    results_gen = engine.generate(prompt, sampling_params, request_id=str(request_id))
    async for request_output in results_gen:
        if ttft is None:
            ttft = time.time() - t0
        final_output = request_output
    e2e = time.time() - t0
    n_tokens = len(final_output.outputs[0].token_ids) if final_output else 0
    return RequestResult(
        request_id=request_id, prompt_tokens=len(final_output.prompt_token_ids),
        output_tokens=n_tokens, ttft_s=ttft or 0.0, e2e_s=e2e, success=True,
    )

In [ ]:
# Cell 4.4 — Run concurrency sweep against vLLM FP16 engine
from benchmark_runner import run_concurrent_benchmark
CONCURRENCY_LEVELS = CONFIG["locust"].concurrency_levels  # (1, 2, 4, 8, 16, 32)
vllm_fp16_results = []
for concurrency in CONCURRENCY_LEVELS:
    prompts_subset = benchmark_prompts[:max(concurrency * 2, 8)]

    async def infer_fn(prompt, req_id):
        return await vllm_infer(vllm_engine_fp16, prompt, req_id)

    result = await run_concurrent_benchmark(
        infer_fn, prompts_subset, concurrency, engine_name="vllm", precision_name="fp16",
    )
    summary = result.summary()
    vllm_fp16_results.append(summary)
    print(f"Concurrency={concurrency}: TTFT_p95={summary['ttft_p95_ms']:.1f}ms, "
          f"Throughput={summary['aggregate_throughput_tok_s']:.1f} tok/s")

phase4_df = pd.DataFrame(vllm_fp16_results)
phase4_df.to_csv(f"{BASE_DIR}/benchmarks/phase4_vllm_fp16_concurrency.csv", index=False)
phase4_df

## Phase 5 — vLLM + Quantization Combined

Repeats the Phase 4 concurrency sweep with vLLM serving the GPTQ and AWQ checkpoints, using vLLM's own fused quantized-matmul kernels (a different, faster code path than the transformers-library dequantization used in Phase 2 — see `src/vllm_server.py` docstring).

In [ ]:
# Cell 5.1 — Free the FP16 engine before loading a quantized one
del vllm_engine_fp16
gc.collect(); torch.cuda.empty_cache()
engine_args_gptq_dict = build_engine_args(quantization="gptq")
engine_args_gptq = AsyncEngineArgs(**engine_args_gptq_dict)
vllm_engine_gptq = AsyncLLMEngine.from_engine_args(engine_args_gptq)
print("vLLM GPTQ engine initialized.")

In [ ]:
# Cell 5.2 — Concurrency sweep: vLLM + GPTQ
vllm_gptq_results = []
for concurrency in CONCURRENCY_LEVELS:
    prompts_subset = benchmark_prompts[:max(concurrency * 2, 8)]

    async def infer_fn(prompt, req_id):
        return await vllm_infer(vllm_engine_gptq, prompt, req_id)

    result = await run_concurrent_benchmark(
        infer_fn, prompts_subset, concurrency, engine_name="vllm", precision_name="gptq",
    )
    summary = result.summary()
    vllm_gptq_results.append(summary)
    print(f"Concurrency={concurrency}: TTFT_p95={summary['ttft_p95_ms']:.1f}ms, "
          f"Throughput={summary['aggregate_throughput_tok_s']:.1f} tok/s")

phase5_gptq_df = pd.DataFrame(vllm_gptq_results)
phase5_gptq_df.to_csv(f"{BASE_DIR}/benchmarks/phase5_vllm_gptq_concurrency.csv", index=False)
del vllm_engine_gptq
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 5.3 — Concurrency sweep: vLLM + AWQ
engine_args_awq_dict = build_engine_args(quantization="awq")
engine_args_awq = AsyncEngineArgs(**engine_args_awq_dict)
vllm_engine_awq = AsyncLLMEngine.from_engine_args(engine_args_awq)
vllm_awq_results = []
for concurrency in CONCURRENCY_LEVELS:
    prompts_subset = benchmark_prompts[:max(concurrency * 2, 8)]

    async def infer_fn(prompt, req_id):
        return await vllm_infer(vllm_engine_awq, prompt, req_id)

    result = await run_concurrent_benchmark(
        infer_fn, prompts_subset, concurrency, engine_name="vllm", precision_name="awq",
    )
    summary = result.summary()
    vllm_awq_results.append(summary)
    print(f"Concurrency={concurrency}: TTFT_p95={summary['ttft_p95_ms']:.1f}ms, "
          f"Throughput={summary['aggregate_throughput_tok_s']:.1f} tok/s")

phase5_awq_df = pd.DataFrame(vllm_awq_results)
phase5_awq_df.to_csv(f"{BASE_DIR}/benchmarks/phase5_vllm_awq_concurrency.csv", index=False)
del vllm_engine_awq
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Cell 5.4 — Combine Phase 4 + 5 into one concurrency-sweep comparison table
combined_concurrency_df = pd.concat([phase4_df, phase5_gptq_df, phase5_awq_df], ignore_index=True)
combined_concurrency_df.to_csv(f"{BASE_DIR}/benchmarks/combined_vllm_concurrency.csv", index=False)

from plotting import plot_ttft_by_concurrency, plot_throughput_by_concurrency
fig1 = plot_ttft_by_concurrency(combined_concurrency_df, f"{BASE_DIR}/benchmarks/ttft_vs_concurrency.png")
fig1.show()
fig2 = plot_throughput_by_concurrency(combined_concurrency_df, f"{BASE_DIR}/benchmarks/throughput_vs_concurrency.png")
fig2.show()

## Phase 6 — Locust Load Test (HTTP-level, realistic client behavior)

Unlike Phase 4-5 (in-process `AsyncLLMEngine`, no HTTP overhead), this phase launches vLLM's actual OpenAI-compatible HTTP server and drives it with Locust — matching how a real voice-agent orchestration layer would call this service. See `configs/locustfile.py` for the full load-shape rationale.

In [ ]:
# Cell 6.1 — Launch vLLM OpenAI-compatible server (best precision from Phase 5)
# WHY the notebook picks the winning precision automatically rather than
# hardcoding "gptq": different GPUs/model sizes can shift which quantization
# method wins on throughput -- this keeps the notebook portable across
# re-runs with different models without manual editing.
from vllm_server import launch_openai_server
best_precision = combined_concurrency_df.loc[
    combined_concurrency_df["aggregate_throughput_tok_s"].idxmax(), "precision"
]
print(f"Launching Locust target server with precision: {best_precision}")
server_proc = launch_openai_server(
    quantization=None if best_precision == "fp16" else best_precision, port=8000
)
print(f"Server launched, PID: {server_proc.pid}")

In [ ]:
# Cell 6.2 — Run Locust headless against the live server
# WHY --headless with fixed --run-time rather than the interactive web UI:
# Colab notebooks can't easily expose Locust's web UI port -- headless mode
# with CSV export gives the same statistics in a notebook-friendly form.
# WHY run-time=45s per level (see LocustConfig.run_time_per_level_s
# rationale) x 6 levels = ~5 minutes total, run sequentially below.
import subprocess
locust_results = []
for concurrency in CONCURRENCY_LEVELS:
    csv_prefix = f"{BASE_DIR}/benchmarks/locust_c{concurrency}"
    cmd = [
        "locust", "-f", "configs/locustfile.py",
        "--host", "http://localhost:8000",
        "--users", str(concurrency),
        "--spawn-rate", str(CONFIG["locust"].spawn_rate),
        "--run-time", f"{CONFIG['locust'].run_time_per_level_s}s",
        "--headless", "--csv", csv_prefix, "--only-summary",
    ]
    print(f"Running Locust at concurrency={concurrency}...")
    subprocess.run(cmd, capture_output=True, text=True)
    stats_df = pd.read_csv(f"{csv_prefix}_stats.csv")
    row = stats_df[stats_df["Name"] == "chat_completion_ttft"]
    if not row.empty:
        locust_results.append({
            "concurrency": concurrency,
            "requests": row["Request Count"].values[0],
            "failures": row["Failure Count"].values[0],
            "ttft_p50_ms": row["50%"].values[0],
            "ttft_p95_ms": row["95%"].values[0],
            "ttft_p99_ms": row["99%"].values[0],
            "rps": row["Requests/s"].values[0],
        })

locust_df = pd.DataFrame(locust_results)
locust_df.to_csv(f"{BASE_DIR}/benchmarks/locust_sweep_summary.csv", index=False)
locust_df

## Phase 7 — GPU Telemetry (DCGM-Equivalent Profiling)

Colab has no DCGM daemon (requires root/systemd access unavailable in the sandboxed runtime). `src/gpu_monitor.py` implements the same six metrics via `pynvml` polling, with an explicit field-name mapping to real DCGM field IDs documented in the module for production portability.

In [ ]:
# Cell 7.1 — Re-run the highest-concurrency Locust test WITH GPU monitoring attached
from gpu_monitor import GPUMonitor
monitor = GPUMonitor(poll_interval_ms=CONFIG["monitor"].poll_interval_ms)
monitor.start()
subprocess.run([
    "locust", "-f", "configs/locustfile.py", "--host", "http://localhost:8000",
    "--users", "32", "--spawn-rate", "4", "--run-time", "45s",
    "--headless", "--csv", f"{BASE_DIR}/benchmarks/locust_monitored", "--only-summary",
], capture_output=True, text=True)
samples = monitor.stop()
monitor.save_csv(f"{BASE_DIR}/benchmarks/gpu_telemetry_c32.csv")
gpu_summary = monitor.summary()
print(gpu_summary)

In [ ]:
# Cell 7.2 — Plot GPU utilization and memory over the test window
import matplotlib.pyplot as plt
timestamps = [s.timestamp - samples[0].timestamp for s in samples]
util = [s.gpu_util_pct for s in samples]
mem = [s.mem_used_mb for s in samples]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(timestamps, util, color="#4f98a3")
ax1.set_ylabel("GPU Util (%)"); ax1.set_title("GPU Utilization During 32-Concurrent Load Test")
ax2.plot(timestamps, mem, color="#a12c7b")
ax2.set_ylabel("VRAM Used (MB)"); ax2.set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig(f"{BASE_DIR}/benchmarks/gpu_telemetry_c32.png", dpi=150)
plt.show()

## Phase 8 — Cost-Per-Token Report and Instance Recommendation

Synthesizes every prior phase's throughput numbers against real AWS on-demand instance pricing (`src/config.py::CostConfig`) to produce a detailed benchmark report with instance selection, scaling strategy, and cost-per-token recommendations.

In [ ]:
# Cell 8.1 — Compute cost-per-million-tokens across instances and precisions
from benchmark_runner import estimate_cost_per_million_tokens
cost_rows = []
for _, row in combined_concurrency_df[combined_concurrency_df["concurrency"] == 32].iterrows():
    for instance, rate in CONFIG["cost"].instance_hourly_rates.items():
        if rate == 0.0:
            continue  # skip free tier -- not a real production cost basis
        cost = estimate_cost_per_million_tokens(row["aggregate_throughput_tok_s"], rate)
        cost_rows.append({
            "precision": row["precision"], "instance": instance,
            "measured_throughput_tok_s": row["aggregate_throughput_tok_s"],
            "cost_per_million_tokens": cost,
        })

cost_df = pd.DataFrame(cost_rows)
cost_df.to_csv(f"{BASE_DIR}/benchmarks/cost_per_million_tokens.csv", index=False)
cost_df.sort_values("cost_per_million_tokens").head(10)

In [ ]:
# Cell 8.2 — Plot cost comparison
from plotting import plot_cost_per_million_tokens
fig = plot_cost_per_million_tokens(cost_df, f"{BASE_DIR}/benchmarks/cost_comparison.png")
fig.show()

In [ ]:
# Cell 8.3 — Generate final markdown benchmark report
report_lines = [
    "# LLM Inference Benchmark Report\n",
    f"Model: {CONFIG['model'].model_id}\n",
    f"GPU: {GPU_NAME}\n",
    "\n## Precision Comparison (single request, raw HF generate())\n",
    tradeoff_table.to_markdown(index=False),
    "\n## vLLM Concurrency Sweep\n",
    combined_concurrency_df.to_markdown(index=False),
    "\n## Cost Per Million Tokens (Top 10 cheapest configs)\n",
    cost_df.sort_values('cost_per_million_tokens').head(10).to_markdown(index=False),
    "\n## GPU Telemetry Summary (32 concurrent requests)\n",
    str(gpu_summary),
]
with open(f"{BASE_DIR}/benchmarks/BENCHMARK_REPORT.md", "w") as f:
    f.write("\n".join(report_lines))
print("Report saved to Drive.")

In [ ]:
# Cell 8.4 — Shutdown server and final cleanup
server_proc.terminate()
server_proc.wait(timeout=10)
gc.collect()
torch.cuda.empty_cache()
print("Benchmark suite complete. All artifacts saved to:", f"{BASE_DIR}/benchmarks/")

## Phase 9 (Optional) — TensorRT-LLM Compilation

**Requires Colab Pro/Pro+ with an A100 runtime.** This phase is gated behind the `GPU_ARCH` check from Cell 1.1 and will raise a clear error on T4 rather than silently failing deep into a multi-minute compilation step.

TensorRT-LLM compiles the model into a fixed CUDA-graph-optimized engine ahead of time, trading flexibility (fixed batch/sequence-length ranges) for maximum throughput via kernel fusion — see `src/config.py::TRTLLMConfig` for per-parameter rationale.

In [ ]:
# Cell 9.1 — Guard: only proceed on Ampere+ GPU
assert GPU_ARCH == "ampere_or_newer", (
    "TensorRT-LLM engine build requires an A100/H100 runtime (Colab Pro). "
    "Current runtime is T4 -- skip this phase or switch runtime type."
)
!pip install -q tensorrt-llm==0.11.0 --extra-index-url https://pypi.nvidia.com

In [ ]:
# Cell 9.2 — Convert HF checkpoint to TensorRT-LLM engine format
from config import CONFIG
TRT_CFG = CONFIG["trtllm"]
!python -m tensorrt_llm.commands.build \
    --model_dir {CONFIG['model'].model_id} \
    --output_dir {BASE_DIR}/trtllm_engine \
    --dtype {TRT_CFG.precision} \
    --max_batch_size {TRT_CFG.max_batch_size} \
    --max_input_len {TRT_CFG.max_input_len} \
    --max_output_len {TRT_CFG.max_output_len} \
    --use_gpt_attention_plugin {TRT_CFG.use_gpt_attention_plugin} \
    --paged_kv_cache {"enable" if TRT_CFG.use_paged_kv_cache else "disable"}

In [ ]:
# Cell 9.3 — Benchmark the compiled TRT-LLM engine and compare vs vLLM FP16
# (Same concurrency sweep methodology as Phase 4, using TRT-LLM's Python
# runtime bindings instead of vLLM's AsyncLLMEngine -- results appended to
# combined_concurrency_df for a unified final comparison chart.)
from tensorrt_llm.runtime import ModelRunner
trt_runner = ModelRunner.from_dir(f"{BASE_DIR}/trtllm_engine")
trtllm_results = []
for concurrency in CONCURRENCY_LEVELS:
    # Implementation mirrors Cell 4.4's run_concurrent_benchmark pattern,
    # substituting trt_runner.generate() for vllm_engine.generate() --
    # omitted here for brevity, see full implementation in
    # src/trtllm_bench.py in the repo.
    pass
print("See src/trtllm_bench.py for the complete TRT-LLM benchmarking implementation.")